# Лабораторная работа №5 — градиентный бустинг

## 1. Выбор начальных условий

- **Spambase:** по 57 признакам письма определяем, спам ли оно. Основная метрика F1-macro одинаково учитывает обе метки; accuracy показывает долю верных ответов.
- **SML2010:** по 18 текущим признакам прогнозируем температуру в столовой через 15 минут. MAE показывает среднюю абсолютную ошибку в °C, а RMSE сильнее штрафует крупные ошибки.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from data_setup import DINING_TEMPERATURE, RANDOM_STATE, load_sml2010, load_spambase, sml2010_cv, spambase_cv
from notebooks.tree import DecisionTreeNumPy

Xc_train, Xc_test, yc_train, yc_test = load_spambase()
Xr_train, Xr_test, yr_train, yr_test = load_sml2010()
print(f"Spambase: {Xc_train.shape} / {Xc_test.shape}")
print(f"SML2010:  {Xr_train.shape} / {Xr_test.shape}")

Spambase: (3363, 57) / (841, 57)
SML2010:  (2763, 18) / (1370, 18)


## 2. Создание бейзлайна и оценка качества

Обучаем градиентный бустинг sklearn с базовыми настройками: 100 деревьев глубины 3, скорость обучения 0,1. В классификации используется логистическая потеря, в регрессии — квадратичная. Сравниваем тестовые метрики; для температуры отдельно показываем ориентир — прогноз по текущему измерению.

In [2]:
def classification_metrics(y, prediction):
    return {"f1_macro": f1_score(y, prediction, average="macro"), "accuracy": accuracy_score(y, prediction)}


def regression_metrics(y, prediction):
    return {"mae": mean_absolute_error(y, prediction), "rmse": root_mean_squared_error(y, prediction)}


clf = GradientBoostingClassifier(random_state=RANDOM_STATE)
reg = GradientBoostingRegressor(random_state=RANDOM_STATE)
clf.fit(Xc_train, yc_train)
reg.fit(Xr_train, yr_train)
classification_scores = {"sklearn_base": classification_metrics(yc_test, clf.predict(Xc_test))}
regression_scores = {"sklearn_base": regression_metrics(yr_test, reg.predict(Xr_test))}
persistence = regression_metrics(yr_test, Xr_test[DINING_TEMPERATURE])

display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict({**regression_scores, "текущая температура": persistence}, orient="index"))

,f1_macro,accuracy
sklearn_base,0.942768,0.945303


,mae,rmse
sklearn_base,0.309688,0.685667
текущая температура,0.116360,0.133181


## 3. Улучшение бейзлайна

### Гипотезы

1. Уменьшение глубины с 3 до 2 может снизить переобучение отдельных деревьев.
2. Меньший шаг (`0,05` вместо `0,1`) или меньшее число деревьев (`50` вместо `100`) может улучшить качество. Проверяем эти параметры вместе: меньший шаг обычно требует больше деревьев.

### Проверка гипотез

Перебираем восемь конфигураций отдельно для каждой задачи на обучающей части. Сохраняем стратифицированную валидацию и F1-macro для Spambase, временные фолды с зазором и MAE для SML2010.

In [3]:
grid = {"max_depth": [2, 3], "learning_rate": [0.05, 0.1], "n_estimators": [50, 100]}
clf_search = GridSearchCV(clf, grid, scoring="f1_macro", cv=spambase_cv()).fit(Xc_train, yc_train)
reg_search = GridSearchCV(reg, grid, scoring="neg_mean_absolute_error", cv=sml2010_cv()).fit(Xr_train, yr_train)


def cv_table(search, metric, sign=1):
    results = search.cv_results_
    table = pd.DataFrame(results["params"])
    table[metric] = sign * results["mean_test_score"]
    return table.sort_values(metric, ascending=(sign == -1))


display(cv_table(clf_search, "CV F1-macro"))
display(cv_table(reg_search, "CV MAE, °C", sign=-1))

,learning_rate,max_depth,n_estimators,CV F1-macro
7,0.10,3,100,0.939377
5,0.10,2,100,0.935223
3,0.05,3,100,0.934216
6,0.10,3,50,0.932367
1,0.05,2,100,0.928160
4,0.10,2,50,0.928155
2,0.05,3,50,0.921006
0,0.05,2,50,0.911107


,learning_rate,max_depth,n_estimators,"CV MAE, °C"
7,0.10,3,100,0.133039
6,0.10,3,50,0.142757
3,0.05,3,100,0.142963
5,0.10,2,100,0.153187
1,0.05,2,100,0.173881
4,0.10,2,50,0.180576
2,0.05,3,50,0.311872
0,0.05,2,50,0.390021


### Улучшенный бейзлайн

Берём лучшую среднюю валидационную конфигурацию каждой задачи и оцениваем переобученные модели на прежнем тесте. Сопоставляем результат с пунктом 2.

In [4]:
print("Spambase:", clf_search.best_params_)
print("SML2010:", reg_search.best_params_)
classification_scores["sklearn_improved"] = classification_metrics(yc_test, clf_search.predict(Xc_test))
regression_scores["sklearn_improved"] = regression_metrics(yr_test, reg_search.predict(Xr_test))
display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict(regression_scores, orient="index"))

Spambase: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
SML2010: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}


,f1_macro,accuracy
sklearn_base,0.942768,0.945303
sklearn_improved,0.942768,0.945303


,mae,rmse
sklearn_base,0.309688,0.685667
sklearn_improved,0.309688,0.685667


**Выводы.** Среди проверенных конфигураций для обеих задач лучшей по CV осталась базовая: 100 деревьев глубины 3 и шаг 0,1. Поэтому тестовые метрики не изменились: F1-macro 0,9428 для Spambase и MAE 0,3097 °C для SML2010. Прогноз по текущей температуре точнее (MAE 0,1164 °C).

## 4. Имплементация алгоритма машинного обучения

###  Базовая реализация

Используем [ядро дерева](tree.py). В регрессии начинаем со среднего `y`, строим очередное дерево на остатках `y − ŷ` и добавляем его прогноз с коэффициентом `learning_rate`. В классификации начинаем с логита доли класса `1`; на каждом шаге дерево приближает `y − sigmoid(logit)`, после чего обновляем логит. Оба варианта обучают **регрессионные** деревья на вещественных остатках.

Классификатор учебный: в листьях оставляем средние остатки, без отдельного ньютоновского обновления значений листьев, которое делает sklearn. Поэтому одинаковые гиперпараметры не обязаны давать одинаковые прогнозы.

In [5]:
class GradientBoostingNumPy:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, classification=True):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.classification = classification

    @staticmethod
    def sigmoid(logits):
        return 1 / (1 + np.exp(-np.clip(logits, -50, 50)))

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        self.initial = y.mean()
        if self.classification:
            p = np.clip(self.initial, 1e-12, 1 - 1e-12)
            self.initial = np.log(p / (1 - p))
        prediction = np.full(len(y), self.initial)
        self.trees = []
        for _ in range(self.n_estimators):
            fitted = self.sigmoid(prediction) if self.classification else prediction
            tree = DecisionTreeNumPy(classification=False, max_depth=self.max_depth).fit(X, y - fitted)
            prediction += self.learning_rate * tree.predict(X)
            self.trees.append(tree)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        prediction = np.full(len(X), self.initial)
        for tree in self.trees:
            prediction += self.learning_rate * tree.predict(X)
        return (prediction > 0).astype(int) if self.classification else prediction

Проверяем два шага на маленькой регрессии, где остатки можно посчитать вручную, и первый шаг бинарной классификации. Затем обучаем собственный бустинг с базовыми параметрами и сравниваем со sklearn из пункта 2.

In [6]:
X_toy = [[0], [1], [2], [3]]
toy_reg = GradientBoostingNumPy(n_estimators=2, learning_rate=0.5, max_depth=1, classification=False)
np.testing.assert_allclose(toy_reg.fit(X_toy, [0, 0, 2, 2]).predict(X_toy), [0.25, 0.25, 1.75, 1.75])
toy_clf = GradientBoostingNumPy(n_estimators=1, max_depth=1)
np.testing.assert_array_equal(toy_clf.fit(X_toy, [0, 0, 1, 1]).predict(X_toy), [0, 0, 1, 1])

clf_own = GradientBoostingNumPy().fit(Xc_train, yc_train)
reg_own = GradientBoostingNumPy(classification=False).fit(Xr_train, yr_train)
classification_scores["custom_base"] = classification_metrics(yc_test, clf_own.predict(Xc_test))
regression_scores["custom_base"] = regression_metrics(yr_test, reg_own.predict(Xr_test))
base_variants = ["sklearn_base", "custom_base"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[base_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[base_variants])

,f1_macro,accuracy
sklearn_base,0.942768,0.945303
custom_base,0.914550,0.919144


,mae,rmse
sklearn_base,0.309688,0.685667
custom_base,0.309811,0.684882


**Сравнение с пунктом 2.** Базовый NumPy-классификатор получил F1-macro 0,9145 против 0,9428 у sklearn. Он использует только шаг по антиградиенту, без поправки значений листьев. Для регрессии результаты близки: MAE 0,3098 °C у NumPy и 0,3097 °C у sklearn.

### 4.2. Улучшенная реализация

Переносим число деревьев, их глубину и скорость обучения из пункта 3. Не подбираем параметры заново. Сравниваем собственные модели с улучшенными sklearn на том же тесте.

In [7]:
clf_own_improved = GradientBoostingNumPy(**clf_search.best_params_).fit(Xc_train, yc_train)
reg_own_improved = GradientBoostingNumPy(
    classification=False, **reg_search.best_params_,
).fit(Xr_train, yr_train)
classification_scores["custom_improved"] = classification_metrics(yc_test, clf_own_improved.predict(Xc_test))
regression_scores["custom_improved"] = regression_metrics(yr_test, reg_own_improved.predict(Xr_test))
improved_variants = ["sklearn_improved", "custom_improved"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[improved_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[improved_variants])

,f1_macro,accuracy
sklearn_improved,0.942768,0.945303
custom_improved,0.914550,0.919144


,mae,rmse
sklearn_improved,0.309688,0.685667
custom_improved,0.309811,0.684882


**Сравнение с пунктом 3.** Перенесённые настройки совпали с базовыми, поэтому у обеих реализаций метрики остались неизменными. На этом тесте от перебора пользы нет; вывод ограничен проверенной сеткой.


In [8]:
variants = ["sklearn_base", "sklearn_improved", "custom_base", "custom_improved"]
rows = []
for task, scores in [("classification", classification_scores), ("regression", regression_scores)]:
    for variant in variants:
        rows.append({"lab": 5, "task": task, "algorithm": "GradientBoosting", "variant": variant, **scores[variant]})

results = pd.DataFrame(rows)
results.to_csv(root / "results" / "lab05.csv", index=False, float_format="%.8f")
for task in results["task"].unique():
    display(results.loc[results["task"] == task].dropna(axis=1, how="all").reset_index(drop=True))

,lab,task,algorithm,variant,f1_macro,accuracy
0,5,classification,GradientBoosting,sklearn_base,0.942768,0.945303
1,5,classification,GradientBoosting,sklearn_improved,0.942768,0.945303
2,5,classification,GradientBoosting,custom_base,0.914550,0.919144
3,5,classification,GradientBoosting,custom_improved,0.914550,0.919144


,lab,task,algorithm,variant,mae,rmse
0,5,regression,GradientBoosting,sklearn_base,0.309688,0.685667
1,5,regression,GradientBoosting,sklearn_improved,0.309688,0.685667
2,5,regression,GradientBoosting,custom_base,0.309811,0.684882
3,5,regression,GradientBoosting,custom_improved,0.309811,0.684882
